# JadeDragon — Train YOLOv8m on LGRC2024

Reproduces the LGRC2024 paper's baseline (YOLOv8m, 28 classes, 150 epochs, AdamW, lr=0.001).

**Setup before running**:
1. Upload the `LGRC2024 dataset/` folder to Google Drive (e.g., `MyDrive/JadeDragon/LGRC2024 dataset/`).
2. Upload this whole project repo to Drive too (e.g., `MyDrive/JadeDragon/JadeDragon Transcriber/`).
3. Runtime → Change runtime type → GPU (T4 free, A100/L4 paid).

Expected runtime on T4: ~1.5–3 hours for 150 epochs.

In [ ]:
# 1. Mount Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/JadeDragon/JadeDragon Transcriber'
DATASET = '/content/drive/MyDrive/JadeDragon/LGRC2024 dataset/datasets'
import os
assert os.path.isdir(PROJECT), f'Project not found at {PROJECT}'
assert os.path.isdir(DATASET), f'Dataset not found at {DATASET}'
%cd {PROJECT}

In [ ]:
# 2. Install deps. Ultralytics is the only one strictly required for training.
!pip install -q ultralytics music21

In [ ]:
# 3. Verify label indexing and (if needed) remap to 0-indexed
!python scripts/prepare_dataset.py --src "{DATASET}" --dst /content/lgrc2024_yolo --remap --symlink

In [ ]:
# 4. Build a Colab-local data.yaml that points at the remapped tree
data_yaml = '''\
path: /content/lgrc2024_yolo
train: images/train
val: images/val
nc: 28
names:
  0: keySignature-bB
  1: keySignature-C
  2: keySignature-D
  3: keySignature-bE
  4: keySignature-F
  5: keySignature-G
  6: keySignature-A
  7: note-C3
  8: note-D3
  9: note-E3
  10: note-F3
  11: note-G3
  12: note-A3
  13: note-B3
  14: note-C4
  15: note-D4
  16: note-E4
  17: note-F4
  18: note-G4
  19: note-A4
  20: note-B4
  21: note-C5
  22: note-D5
  23: note-E5
  24: note-F5
  25: note-G5
  26: note-A5
  27: note-B5
'''
with open('/content/lgrc2024.yaml', 'w') as f:
    f.write(data_yaml)

In [ ]:
# 5. Train YOLOv8m. Disable spatial augmentations (rotate/flip) — they break
#    gongche reading order. Keep mild color jitter for paper-color robustness.
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/lgrc2024.yaml',
    epochs=150,
    batch=16,
    imgsz=640,
    lr0=0.001,
    optimizer='AdamW',
    name='lgrc2024_yolov8m',
    device=0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0,
    flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
)

In [ ]:
# 6. Copy the best weights back to Drive so they persist between sessions
import shutil, os
src = 'runs/detect/lgrc2024_yolov8m/weights/best.pt'
dst_dir = f'{PROJECT}/checkpoints'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/lgrc2024_yolov8m_best.pt')
print(f'Saved to {dst_dir}/lgrc2024_yolov8m_best.pt')

In [ ]:
# 7. Quick eval on val set
metrics = model.val()
print(metrics)

Once training is done, open `02_demo_one_page_e2e.ipynb` to run the full pipeline
(image → MusicXML + audio) using this trained checkpoint.